# E-commerce Purchase Prediction
Predicting 7-day purchases from 42M October and 67M November events.

## 0. Setup
The heavy CSV processing runs only when RUN_DATA_PREP is True. Otherwise, the notebook reads the cleaned Parquet files from a saved Kaggle dataset.

In [1]:
!pip install -q duckdb

import os
import duckdb

# Set to True only if the cleaned Parquet files ever need to be rebuilt from the raw CSVs
RUN_DATA_PREP = False

con = duckdb.connect()
con.sql("SET memory_limit='20GB'")
con.sql("SET threads=4")
con.sql("SET temp_directory='/tmp/duckdb'")

def find_file(name):
    # Look in this session's output first, then in attached datasets
    for base in ['/kaggle/working', '/kaggle/input']:
        for root, _, files in os.walk(base):
            if name in files:
                return os.path.join(root, name)
    return None

# Quick check: should print the paths of the three files from your new dataset
print(find_file('oct.parquet'))
print(find_file('oct_clean.parquet'))
print(find_file('nov_clean.parquet'))

/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct.parquet
/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct_clean.parquet
/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/nov_clean.parquet


In [2]:
# Check that the dataset copies exist in /kaggle/input
for name in ['oct.parquet', 'oct_clean.parquet', 'nov_clean.parquet']:
    found = [os.path.join(r, name) for r, _, f in os.walk('/kaggle/input') if name in f]
    print(name, found)

oct.parquet ['/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct.parquet']
oct_clean.parquet ['/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct_clean.parquet']
nov_clean.parquet ['/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/nov_clean.parquet']


In [3]:
# Convert the raw October CSV to Parquet (runs only when rebuilding data)
if RUN_DATA_PREP:
    src = find_file('2019-Oct.csv')
    con.sql(f"""
        COPY (SELECT * FROM read_csv_auto('{src}'))
        TO '/kaggle/working/oct.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print('October converted')

In [4]:
con.sql(f"CREATE OR REPLACE VIEW oct AS SELECT * FROM '{find_file('oct.parquet')}'")

In [5]:
# Remove exact duplicates and add derived columns (runs only when rebuilding data)
if RUN_DATA_PREP:
    con.sql("""
        COPY (
            SELECT DISTINCT
                event_time,
                CAST(event_time AS DATE)              AS event_date,
                HOUR(event_time)                      AS event_hour,
                DAYOFWEEK(event_time)                 AS day_of_week,
                event_type, product_id, category_id, category_code,
                split_part(category_code, '.', 1)     AS category_l1,
                COALESCE(brand, 'unknown')            AS brand,
                price, user_id, user_session
            FROM oct
        ) TO '/kaggle/working/oct_clean.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

con.sql(f"CREATE OR REPLACE VIEW oct_clean AS SELECT * FROM '{find_file('oct_clean.parquet')}'")
con.sql("SELECT COUNT(*) AS rows FROM oct_clean").df()

,rows
0,42418544


In [6]:
# Clean November with the same rules as October (runs only when rebuilding data)
if RUN_DATA_PREP:
    nov_src = find_file('2019-Nov.csv')
    con.sql(f"""
        COPY (
            SELECT DISTINCT
                event_time,
                CAST(event_time AS DATE)              AS event_date,
                HOUR(event_time)                      AS event_hour,
                DAYOFWEEK(event_time)                 AS day_of_week,
                event_type, product_id, category_id, category_code,
                split_part(category_code, '.', 1)     AS category_l1,
                COALESCE(brand, 'unknown')            AS brand,
                price, user_id, user_session
            FROM read_csv_auto('{nov_src}')
        ) TO '/kaggle/working/nov_clean.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

con.sql(f"CREATE OR REPLACE VIEW nov_clean AS SELECT * FROM '{find_file('nov_clean.parquet')}'")
con.sql("SELECT COUNT(*) AS rows, MIN(event_time) AS first_event, MAX(event_time) AS last_event FROM nov_clean").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_event,last_event
0,67401460,2019-11-01,2019-11-30 23:59:59


In [7]:
# Raw November row count, used to calculate duplicates removed (runs only when rebuilding data)
# Recorded result: 67,501,979 raw rows, 100,519 duplicates removed (0.15%)
if RUN_DATA_PREP:
    con.sql(f"SELECT COUNT(*) AS raw_rows FROM read_csv_auto('{find_file('2019-Nov.csv')}')").df()

## 1. Data loading

In [8]:
!pip install -q duckdb

import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        path = os.path.join(root, f)
        print(path, round(os.path.getsize(path) / 1e9, 2), 'GB')


/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv 9.01 GB
/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv 5.67 GB
/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/nov_clean.parquet 2.44 GB
/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct.parquet 0.75 GB
/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct_clean.parquet 1.46 GB


In [9]:
import duckdb

# Find the October file automatically
src = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == '2019-Oct.csv':
            src = os.path.join(root, f)

print('Found:', src)

con = duckdb.connect()
con.sql("SET memory_limit='20GB'")
con.sql("SET threads=4")
con.sql("SET temp_directory='/tmp/duckdb'")

con.sql(f"""
    COPY (SELECT * FROM read_csv_auto('{src}'))
    TO '/kaggle/working/oct.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print('done')

Found: /kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done


## 2. Data quality audit

In [10]:
con.sql("CREATE OR REPLACE VIEW oct AS SELECT * FROM '/kaggle/working/oct.parquet'")

# 1. Overall scale
con.sql("""
    SELECT COUNT(*) AS rows,
           MIN(event_time) AS first_event,
           MAX(event_time) AS last_event,
           COUNT(DISTINCT user_id) AS users,
           COUNT(DISTINCT user_session) AS sessions,
           COUNT(DISTINCT product_id) AS products
    FROM oct
""").show()

# 2. Event type distribution
con.sql("""
    SELECT event_type, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM oct GROUP BY 1 ORDER BY 2 DESC
""").show()

# 3. Missing values
con.sql("""
    SELECT ROUND(100 * AVG(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END), 2) AS category_null_pct,
           ROUND(100 * AVG(CASE WHEN brand IS NULL THEN 1 ELSE 0 END), 2) AS brand_null_pct,
           ROUND(100 * AVG(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END), 2) AS session_null_pct
    FROM oct
""").show()

# 4. Exact duplicate rows
con.sql("""
    SELECT (SELECT COUNT(*) FROM oct) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM oct)) AS duplicate_rows
""").show()

# 5. Unusual prices
con.sql("""
    SELECT COUNT(*) FILTER (WHERE price <= 0) AS zero_or_negative_price,
           MIN(price) AS min_price, MAX(price) AS max_price
    FROM oct
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬─────────────────────┬─────────────────────┬─────────┬──────────┬──────────┐
│   rows   │     first_event     │     last_event      │  users  │ sessions │ products │
│  int64   │      timestamp      │      timestamp      │  int64  │  int64   │  int64   │
├──────────┼─────────────────────┼─────────────────────┼─────────┼──────────┼──────────┤
│ 42448764 │ 2019-10-01 00:00:00 │ 2019-10-31 23:59:59 │ 3022290 │  9244421 │   166794 │
└──────────┴─────────────────────┴─────────────────────┴─────────┴──────────┴──────────┘

┌────────────┬──────────┬────────┐
│ event_type │    n     │  pct   │
│  varchar   │  int64   │ double │
├────────────┼──────────┼────────┤
│ view       │ 40779399 │  96.07 │
│ cart       │   926516 │   2.18 │
│ purchase   │   742849 │   1.75 │
└────────────┴──────────┴────────┘

┌───────────────────┬────────────────┬──────────────────┐
│ category_null_pct │ brand_null_pct │ session_null_pct │
│      double       │     double     │      double      │
├─────────

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ duplicate_rows │
│     int64      │
├────────────────┤
│          30220 │
└────────────────┘

┌────────────────────────┬───────────┬───────────┐
│ zero_or_negative_price │ min_price │ max_price │
│         int64          │  double   │  double   │
├────────────────────────┼───────────┼───────────┤
│                  68673 │       0.0 │   2574.07 │
└────────────────────────┴───────────┴───────────┘



In [11]:
# A. Zero prices by event type
con.sql("""
    SELECT event_type, COUNT(*) AS zero_price_rows
    FROM oct WHERE price = 0
    GROUP BY 1 ORDER BY 2 DESC
""").show()

# B. Is category_id complete, even where category_code is missing?
con.sql("""
    SELECT ROUND(100 * AVG(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END), 2) AS category_id_null_pct
    FROM oct
""").show()

# C. How many purchases had no cart event in the same session?
con.sql("""
    WITH p AS (SELECT DISTINCT user_session, product_id FROM oct WHERE event_type = 'purchase'),
         c AS (SELECT DISTINCT user_session, product_id FROM oct WHERE event_type = 'cart')
    SELECT COUNT(*) AS purchases,
           COUNT(*) FILTER (WHERE c.user_session IS NULL) AS purchases_without_cart,
           ROUND(100.0 * COUNT(*) FILTER (WHERE c.user_session IS NULL) / COUNT(*), 1) AS pct_without_cart
    FROM p LEFT JOIN c USING (user_session, product_id)
""").show()

┌────────────┬─────────────────┐
│ event_type │ zero_price_rows │
│  varchar   │      int64      │
├────────────┼─────────────────┤
│ view       │           68523 │
│ cart       │             150 │
└────────────┴─────────────────┘

┌──────────────────────┐
│ category_id_null_pct │
│        double        │
├──────────────────────┤
│                  0.0 │
└──────────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────────────────┬──────────────────┐
│ purchases │ purchases_without_cart │ pct_without_cart │
│   int64   │         int64          │      double      │
├───────────┼────────────────────────┼──────────────────┤
│    690372 │                 385432 │             55.8 │
└───────────┴────────────────────────┴──────────────────┘



In [12]:
con.sql("""
    WITH p AS (SELECT DISTINCT user_id, product_id FROM oct WHERE event_type = 'purchase'),
         c AS (SELECT DISTINCT user_id, product_id FROM oct WHERE event_type = 'cart')
    SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE c.user_id IS NULL) / COUNT(*), 1) AS pct_never_carted
    FROM p LEFT JOIN c USING (user_id, product_id)
""").show()

┌──────────────────┐
│ pct_never_carted │
│      double      │
├──────────────────┤
│             53.5 │
└──────────────────┘



## 3. Cleaning decisions
Exact duplicates are removed, zero prices are kept but excluded from price-based features, category_id is used for grouping, and missing brands are labeled "unknown". The same rules are applied to both months.

In [13]:
con.sql("""
    COPY (
        SELECT DISTINCT
            event_time,
            CAST(event_time AS DATE)                 AS event_date,
            HOUR(event_time)                         AS event_hour,
            DAYOFWEEK(event_time)                    AS day_of_week,
            event_type,
            product_id,
            category_id,
            category_code,
            split_part(category_code, '.', 1)        AS category_l1,
            COALESCE(brand, 'unknown')               AS brand,
            price,
            user_id,
            user_session
        FROM oct
    ) TO '/kaggle/working/oct_clean.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.sql("CREATE OR REPLACE VIEW oct_clean AS SELECT * FROM '/kaggle/working/oct_clean.parquet'")
con.sql("SELECT COUNT(*) AS rows FROM oct_clean").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┐
│   rows   │
│  int64   │
├──────────┤
│ 42418544 │
└──────────┘



October 2019 contains 42.4M events from 3.0M users across 9.2M sessions. Views dominate (96%). 55.8% of purchases had no cart event in the same session, so carting is not a required step and a standard view → cart → purchase funnel would be misleading. category_code is missing for 31.8% of events, but category_id is complete and is used for grouping. brand is missing for 14.4% and is labeled “unknown.” 30,220 exact duplicate rows (0.07%) were removed. 68,673 zero-price events are all views or carts, so revenue is unaffected; they are excluded only from price-based features.

55.8% of purchases had no cart event in the same session, and 53.5% of purchased products were never carted by that user during October. Cross-session carting is therefore rare, and carting is not a required step before purchase, so a standard view → cart → purchase funnel would be misleading.

Stage 2: define the prediction problem

## 4. Prediction setup
Training: features from Oct 1–24, label = purchase on Oct 25–31.
Test: features from Oct 8–31, label = purchase on Nov 1–7.
Both feature windows are 24 days, and features never overlap the label window.

In [14]:
# Step 1: build the training labels
con.sql("""
    CREATE OR REPLACE TABLE train_base AS
    WITH feature_users AS (
        SELECT DISTINCT user_id
        FROM oct_clean
        WHERE event_time < TIMESTAMP '2019-10-25'
    ),
    buyers AS (
        SELECT DISTINCT user_id
        FROM oct_clean
        WHERE event_type = 'purchase'
          AND event_time >= TIMESTAMP '2019-10-25'
    )
    SELECT f.user_id,
           CASE WHEN b.user_id IS NOT NULL THEN 1 ELSE 0 END AS purchased_next_7d
    FROM feature_users f
    LEFT JOIN buyers b USING (user_id)
""")

con.sql("""
    SELECT COUNT(*) AS users,
           SUM(purchased_next_7d) AS buyers,
           ROUND(100 * AVG(purchased_next_7d), 2) AS buyer_pct
    FROM train_base
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────┬───────────┐
│  users  │ buyers │ buyer_pct │
│  int64  │ int128 │  double   │
├─────────┼────────┼───────────┤
│ 2559818 │  62181 │      2.43 │
└─────────┴────────┴───────────┘



## 5. Feature engineering

In [15]:
# Step 2: build the features

def build_features(start, cutoff, source='oct_clean'):
    # Only events in [start, cutoff) are used, so no feature can see the label window
    return f"""
    WITH base AS (
        SELECT user_id,
            COUNT(*)                                                  AS n_events,
            COUNT(*) FILTER (WHERE event_type = 'view')               AS n_views,
            COUNT(*) FILTER (WHERE event_type = 'cart')               AS n_carts,
            COUNT(*) FILTER (WHERE event_type = 'purchase')           AS n_purchases,
            COUNT(DISTINCT user_session)                              AS n_sessions,
            COUNT(DISTINCT event_date)                                AS active_days,
            COUNT(DISTINCT product_id)                                AS n_products,
            COUNT(DISTINCT category_id)                               AS n_categories,
            COUNT(DISTINCT brand)                                     AS n_brands,

            -- Recency, in days before the cutoff
            DATE_DIFF('hour', MAX(event_time), TIMESTAMP '{cutoff}') / 24.0                                       AS days_since_last_event,
            DATE_DIFF('hour', MAX(event_time) FILTER (WHERE event_type = 'cart'), TIMESTAMP '{cutoff}') / 24.0     AS days_since_last_cart,
            DATE_DIFF('hour', MAX(event_time) FILTER (WHERE event_type = 'purchase'), TIMESTAMP '{cutoff}') / 24.0 AS days_since_last_purchase,

            -- Recent activity: the 7 days right before the cutoff
            COUNT(*) FILTER (WHERE event_time >= TIMESTAMP '{cutoff}' - INTERVAL 7 DAY)                            AS n_events_last7d,
            COUNT(*) FILTER (WHERE event_type = 'cart' AND event_time >= TIMESTAMP '{cutoff}' - INTERVAL 7 DAY)    AS n_carts_last7d,

            -- Spending; zero prices excluded from price averages (all zero prices are views or carts)
            COALESCE(SUM(price) FILTER (WHERE event_type = 'purchase'), 0)  AS revenue,
            AVG(price) FILTER (WHERE event_type = 'view' AND price > 0)     AS avg_view_price
        FROM {source}
        WHERE event_time >= TIMESTAMP '{start}' AND event_time < TIMESTAMP '{cutoff}'
        GROUP BY user_id
    )
    SELECT *,
        n_carts::DOUBLE     / NULLIF(n_views, 0)    AS cart_per_view,
        n_purchases::DOUBLE / NULLIF(n_views, 0)    AS purchase_per_view,
        n_events::DOUBLE    / NULLIF(n_sessions, 0) AS events_per_session
    FROM base
    """

con.sql(f"""
    CREATE OR REPLACE TABLE train_features AS
    SELECT b.user_id, b.purchased_next_7d, f.* EXCLUDE (user_id)
    FROM train_base b
    JOIN ({build_features('2019-10-01', '2019-10-25')}) f USING (user_id)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
# Step 3: two checks
# Reconciliation: should equal 2,559,818, same as train_base
con.sql("SELECT COUNT(*) AS rows FROM train_features").show()

# Sanity check: do buyers and non-buyers look different?
con.sql("""
    SELECT purchased_next_7d,
           COUNT(*)                                                     AS users,
           ROUND(AVG(n_events), 1)                                      AS avg_events,
           ROUND(AVG(n_sessions), 1)                                    AS avg_sessions,
           ROUND(AVG(days_since_last_event), 1)                         AS avg_days_since_last,
           ROUND(100 * AVG(CASE WHEN n_carts > 0 THEN 1 ELSE 0 END), 1)     AS pct_ever_carted,
           ROUND(100 * AVG(CASE WHEN n_purchases > 0 THEN 1 ELSE 0 END), 1) AS pct_prior_buyers
    FROM train_features
    GROUP BY 1 ORDER BY 1
""").show()

┌─────────┐
│  rows   │
│  int64  │
├─────────┤
│ 2559818 │
└─────────┘

┌───────────────────┬─────────┬────────────┬──────────────┬─────────────────────┬─────────────────┬──────────────────┐
│ purchased_next_7d │  users  │ avg_events │ avg_sessions │ avg_days_since_last │ pct_ever_carted │ pct_prior_buyers │
│       int32       │  int64  │   double   │    double    │       double        │     double      │      double      │
├───────────────────┼─────────┼────────────┼──────────────┼─────────────────────┼─────────────────┼──────────────────┤
│                 0 │ 2497637 │       12.3 │          2.7 │                 9.7 │            10.2 │             10.2 │
│                 1 │   62181 │       39.8 │          6.9 │                 5.5 │            41.9 │             49.3 │
└───────────────────┴─────────┴────────────┴──────────────┴─────────────────────┴─────────────────┴──────────────────┘



In [17]:
# Step 4: prepare November and build the test set

import os

# Find the November file automatically
nov_src = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == '2019-Nov.csv':
            nov_src = os.path.join(root, f)
print('Found:', nov_src)

# Same cleaning as October: remove exact duplicates, add derived columns
con.sql(f"""
    COPY (
        SELECT DISTINCT
            event_time,
            CAST(event_time AS DATE)              AS event_date,
            HOUR(event_time)                      AS event_hour,
            DAYOFWEEK(event_time)                 AS day_of_week,
            event_type, product_id, category_id, category_code,
            split_part(category_code, '.', 1)     AS category_l1,
            COALESCE(brand, 'unknown')            AS brand,
            price, user_id, user_session
        FROM read_csv_auto('{nov_src}')
    ) TO '/kaggle/working/nov_clean.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.sql("CREATE OR REPLACE VIEW nov_clean AS SELECT * FROM '/kaggle/working/nov_clean.parquet'")
con.sql("SELECT COUNT(*) AS rows, MIN(event_time) AS first_event, MAX(event_time) AS last_event FROM nov_clean").df()

Found: /kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_event,last_event
0,67401460,2019-11-01,2019-11-30 23:59:59


### Test set

In [18]:
# Test set: features from Oct 8-31, label = purchase on Nov 1-7
con.sql("""
    CREATE OR REPLACE TABLE test_base AS
    WITH feature_users AS (
        SELECT DISTINCT user_id FROM oct_clean
        WHERE event_time >= TIMESTAMP '2019-10-08'
    ),
    buyers AS (
        SELECT DISTINCT user_id FROM nov_clean
        WHERE event_type = 'purchase'
          AND event_time < TIMESTAMP '2019-11-08'
    )
    SELECT f.user_id,
           CASE WHEN b.user_id IS NOT NULL THEN 1 ELSE 0 END AS purchased_next_7d
    FROM feature_users f
    LEFT JOIN buyers b USING (user_id)
""")

# Same feature function as training, shifted forward by one week
con.sql(f"""
    CREATE OR REPLACE TABLE test_features AS
    SELECT b.user_id, b.purchased_next_7d, f.* EXCLUDE (user_id)
    FROM test_base b
    JOIN ({build_features('2019-10-08', '2019-11-01')}) f USING (user_id)
""")

con.sql("""
    SELECT COUNT(*) AS users,
           SUM(purchased_next_7d) AS buyers,
           ROUND(100 * AVG(purchased_next_7d), 2) AS buyer_pct
    FROM test_features
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,users,buyers,buyer_pct
0,2596037,66974.0,2.58


Buyers are more active, more recent, and far more likely to have carted or bought before. Prior purchase is the strongest single signal, but about half of next-week buyers had never purchased, so the model must also identify first-time buyers from browsing behavior.

In [19]:
con.sql(f"SELECT COUNT(*) AS raw_rows FROM read_csv_auto('{nov_src}')").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,raw_rows
0,67501979
